# Imports

In [168]:
import os
import json
import yaml
from pathlib import Path
import re
import datetime
from collections import defaultdict


import boto3
from dotenv import load_dotenv
import pandas as pd


# Environmental Variables

In [105]:
# set S3_BUCKET environmental variable to arn:aws:s3:::kasbench-test-20260528-377288663341-us-east-1-an
os.environ["S3_BUCKET"] = ""


In [18]:
load_dotenv()
s3_bucket = os.environ.get("S3_BUCKET")
print(f"S3 Bucket: {s3_bucket}")



S3 Bucket: kasbench-test-20260528-377288663341-us-east-1-an


# Constants

In [283]:
run_id = "verification-15-percentile-large-a"
run_id = "mini-verification-15-percentile-large-b"
run_id = "event-verification-medium-a"
s3 = boto3.client("s3")

data_directory = "../data"
output_directory = "../output"



monitored_events_text = """NodeNotReady/TBD/Review
NodeUnreachable/Exogenous/Exclude
NodeHasMemoryPressure/AUT-induced/Retain
NodeHasDiskPressure/AUT-induced/Retain
NodeHasPIDPressure/Infrastructure/Review
Evicted/AUT-Induced/Review
OOMKilling/AUT-Induced/Retain
SystemOOM/TBD/Review
FailedMount/Infrastructure/Exclude
FailedAttachVolume/Infrastructure/Exclude
FailedCreatePodSandBox/Infrastructure/Excluce
FailedPull/Infrastructure/Exclude
ErrImagePull/Infrastructure/Exclude
ImagePullBackOff/Infrastructure/Exclude
NetworkNotReady/Infrastructure/Exclude
FailedScheduling/TBD/Review
Unhealthy/Normal/Retain
BackOff/TBD/Review
CrashLoopBackOff/AUM-Induced/Retain
FailedCreate/TBD/Review
FailedKillPod/TBD/Review
FailedSync/TBD/Review
Rebooted/TBD/Review
FailedGetResourceMetric/AUT-Induced/Retain
FailedComputeMetricsReplicas/AUT-Induced/Retain"""

temp = [event for event in monitored_events_text.split("\n") if event]
monitored_events = {event.split("/")[0]: {"classification": event.split("/")[1], "disposition": event.split("/")[2]} for event in temp}    
monitored_events


{'NodeNotReady': {'classification': 'TBD', 'disposition': 'Review'},
 'NodeUnreachable': {'classification': 'Exogenous', 'disposition': 'Exclude'},
 'NodeHasMemoryPressure': {'classification': 'AUT-induced',
  'disposition': 'Retain'},
 'NodeHasDiskPressure': {'classification': 'AUT-induced',
  'disposition': 'Retain'},
 'NodeHasPIDPressure': {'classification': 'Infrastructure',
  'disposition': 'Review'},
 'Evicted': {'classification': 'AUT-Induced', 'disposition': 'Review'},
 'OOMKilling': {'classification': 'AUT-Induced', 'disposition': 'Retain'},
 'SystemOOM': {'classification': 'TBD', 'disposition': 'Review'},
 'FailedMount': {'classification': 'Infrastructure', 'disposition': 'Exclude'},
 'FailedAttachVolume': {'classification': 'Infrastructure',
  'disposition': 'Exclude'},
 'FailedCreatePodSandBox': {'classification': 'Infrastructure',
  'disposition': 'Excluce'},
 'FailedPull': {'classification': 'Infrastructure', 'disposition': 'Exclude'},
 'ErrImagePull': {'classification': 

# Functions

In [284]:
def download_file_from_s3(s3_bucket, s3_key, local_path):
    """Download a file from S3 to a local path."""
    s3.download_file(s3_bucket, s3_key, local_path)
    print(f"Downloaded {s3_key} to {local_path}")


def get_s3_file_listing(s3_bucket, s3_prefix=''):
    """Get a listing of files in an S3 bucket."""
    file_list = []
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=s3_bucket, Prefix=s3_prefix)
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                file_list.append(obj)
    return file_list

def get_s3_file(s3_bucket, s3_filename, local_filename, data_directory, force=False):
    """Download a file from S3 to a local path."""

    # rename the local file
    local_filename = f"{data_directory}/{local_filename}"

    # skip if file exists and not force
    if os.path.exists(local_filename) and not force: 
        return local_filename
    
    
    # create the local directory, if it doesn't exist 

    dir_path = str(Path(f"{data_directory}/{local_filename}").parent)
    os.makedirs(dir_path, exist_ok=True)

    # download the tile
    s3.download_file(s3_bucket, s3_filename, local_filename)
    return local_filename

def parse_trial(filename):
    pattern = r"^[^/]+/(trial\d{4})/"
    if match := re.match(pattern, filename):
        return match.group(1)
    return None

In [285]:
def get_trial_start_end_times(files, run_id, s3_bucket, data_directory, force=False, verbose=False):
    trials = {}
    for file in files:
        if trial_id := parse_trial((file["Key"])):
            s3_filename = f"{run_id}/{trial_id}/run_details.json"
            local_filename = get_s3_file(s3_bucket, s3_filename, s3_filename, data_directory, force=force)
            with open(local_filename, "r") as f:
                trial = json.load(f)
                start_time = trial["status"]["startTime"] # string in format '2026-09-18T21:18:54.489000+00:00'
                end_time = trial["status"]["endTime"] # string in format '2026-09-18T21:18:54.489000+00:00'
                # convert start_time and end_time to timezone-aware dateteime objects
                start_time = datetime.datetime.fromisoformat(start_time.replace('Z', '+00:00'))
                end_time = datetime.datetime.fromisoformat(end_time.replace('Z', '+00:00'))
                trials[trial_id] = {"start_time": start_time, "end_time": end_time}

    if verbose:
        for trial_id, times in trials.items():
            print(f"{trial_id}: {times['start_time']} -> {times['end_time']}")

    return trials


def get_run_details(s3_bucket, run_id, trial_id, verbose=False):
    """Get the run details from S3"""
    key = f"{run_id}/{trial_id}/run_details.json"
    obj = s3.get_object(Bucket=s3_bucket, Key=key)
    return json.loads(obj['Body'].read())


def get_autoscaler(run_details):
    """ Get the autoscaler from the run details"""
    autoscaler = run_details["initialization"]["autoscaler"]
    return autoscaler    
                

# Code

## Get the listing of files in the S3 bucket for the specified run

In [286]:
files = get_s3_file_listing(s3_bucket, run_id)


## Get the starting and ending time of each trial

In [287]:
trial_times = get_trial_start_end_times(files, run_id, s3_bucket, data_directory, force=False, verbose=False)        
trial_times

{'trial0001': {'start_time': datetime.datetime(2026, 10, 9, 14, 55, 19, 548220, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 10, 9, 15, 25, 25, 356000, tzinfo=datetime.timezone.utc)},
 'trial0002': {'start_time': datetime.datetime(2026, 10, 9, 15, 47, 32, 363742, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 10, 9, 16, 17, 38, 95000, tzinfo=datetime.timezone.utc)}}

## Analyze event logs (warnings.yaml)

In [294]:
def classify_result(result, monitored_events):
    monitored_event = monitored_events.get(result["reason"], {"classification": "Unknown", "disposition": "Unknown"})
    return result | monitored_event


def save_warning_events(s3_bucket, files, trial_times, data_directory, output_directory, exclude_probes=False):
    results = []
    reasons = set()
    counter = 0
    autoscalers = {}
                    
    for file in files:
        if "post/events/warnings.yaml" in file["Key"]:
            filename = file["Key"]
            
            # parse the trial id from the filename
            trial_id = parse_trial(filename)

            # get the autoscaler
            autoscaler = autoscalers.get(trial_id)
            if autoscaler is None:
                run_details = get_run_details(s3_bucket, run_id, trial_id)
                autoscaler = get_autoscaler(run_details)
                autoscalers[trial_id] = autoscaler
                
            # download the event file
            filename = get_s3_file(s3_bucket, filename, filename, data_directory)
            
            # load the event file
            with open(filename, "r") as f:
                events = yaml.load(f, Loader=yaml.FullLoader)
                
            for item in events["items"]:
                reason = item["reason"]
                type = item["type"]
                
                    
                first_timestamp = item["firstTimestamp"] # string in format 2026-09-18T20:39:54Z
                last_timestamp = item["lastTimestamp"] # string in format 2026-09-18T20:39:54Z
                
                
                if first_timestamp is None or last_timestamp is None:
                    continue

                # convert first_timestamp and last_timestamp to timezone aware datetimes
                first_timestamp = datetime.datetime.fromisoformat(first_timestamp.replace('Z', '+00:00'))
                last_timestamp = datetime.datetime.fromisoformat(last_timestamp.replace('Z', '+00:00'))
                
                # exclude events before and after the trial run
                if last_timestamp < trial_times[trial_id]["start_time"] or first_timestamp > trial_times[trial_id]["end_time"]:
                    continue

                reasons.add(reason)
            

                # gather relevant fields
                action = item.get("action")
                kind = item["involvedObject"]["kind"]
                name = item["involvedObject"]["name"]
                namespace = item["involvedObject"].get("namespace")
                message = item["message"]
                type = item["type"]
                count = item["count"]

                # calculate first_timestamp - trial_times[trial_id]["start_time"]
                elapsed_seconds = first_timestamp - trial_times[trial_id]["start_time"]
                minutes, seconds = divmod(elapsed_seconds.total_seconds(), 60)
                # :02.0f ensures 2 digits for minutes, and :02.0f rounds seconds to integers
                elapsed_time_string = f"{int(minutes):02d}:{int(seconds):02d}"    


                result = {
                    "reason": reason,
                    "trial_id": trial_id,
                    "autoscaler": autoscaler,"action": action,
                    
                    "elapsed_time": elapsed_time_string,
                    "type": type,
                    "count": count,
                    "first_timestamp": first_timestamp,
                    "last_timestamp": last_timestamp,
                    "trial_start_time": trial_times[trial_id]["start_time"],
                    "trial_end_time": trial_times[trial_id]["end_time"],
                    "duration": last_timestamp - first_timestamp,
                    "kind": kind,
                    "name": name,
                    "namespace": namespace,
                    "message": message,
                    
                }

                # exclude probe failures in the globeco namespace
                if exclude_probes and namespace == "globeco" and re.search("^(Readiness|Startup|Liveness) probe failed", message):
                    continue

                counter += 1

                result = classify_result(result, monitored_events)
                
                results.append(result)

    # write results array to a json file in the output directory
    with open(f"{output_directory}/audit_results.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
    
    return results, counter

results, count = save_warning_events(s3_bucket, files, trial_times, data_directory, output_directory, exclude_probes=True)

In [295]:
df = pd.DataFrame(results)
df = df.sort_values(by=["first_timestamp"])

# display all the rows of the dataframe
with pd.option_context('display.max_rows', None):
    display(df)

# df

,reason,trial_id,autoscaler,action,elapsed_time,type,count,first_timestamp,last_timestamp,trial_start_time,trial_end_time,duration,kind,name,namespace,message,classification,disposition
0,FailedGetResourceMetric,trial0001,hpa,None,-3:40,Warning,15,2026-10-09 14:53:00+00:00,2026-10-09 14:56:46+00:00,2026-10-09 14:55:19.548220+00:00,2026-10-09 15:25:25.356000+00:00,0 days 00:03:46,HorizontalPodAutoscaler,globeco-fix-engine-hpa,globeco,failed to get cpu utilization: did not receive...,AUT-Induced,Retain
3,FailedGetResourceMetric,trial0001,hpa,None,13:57,Warning,1,2026-10-09 15:09:17+00:00,2026-10-09 15:09:17+00:00,2026-10-09 14:55:19.548220+00:00,2026-10-09 15:25:25.356000+00:00,0 days 00:00:00,HorizontalPodAutoscaler,globeco-portfolio-service-hpa,globeco,failed to get cpu utilization: did not receive...,AUT-Induced,Retain
4,FailedGetResourceMetric,trial0001,hpa,None,13:57,Warning,1,2026-10-09 15:09:17+00:00,2026-10-09 15:09:17+00:00,2026-10-09 14:55:19.548220+00:00,2026-10-09 15:25:25.356000+00:00,0 days 00:00:00,HorizontalPodAutoscaler,globeco-portfolio-service-hpa,globeco,failed to get memory utilization: did not rece...,AUT-Induced,Retain
5,FailedComputeMetricsReplicas,trial0001,hpa,None,13:57,Warning,1,2026-10-09 15:09:17+00:00,2026-10-09 15:09:17+00:00,2026-10-09 14:55:19.548220+00:00,2026-10-09 15:25:25.356000+00:00,0 days 00:00:00,HorizontalPodAutoscaler,globeco-portfolio-service-hpa,globeco,"invalid metrics (2 invalid out of 2), first er...",AUT-Induced,Retain
2,FailedScheduling,trial0001,hpa,None,14:57,Warning,6,2026-10-09 15:10:17+00:00,2026-10-09 15:20:18+00:00,2026-10-09 14:55:19.548220+00:00,2026-10-09 15:25:25.356000+00:00,0 days 00:10:01,Pod,globeco-portfolio-service-8465f76dff-mdtf7,globeco,0/3 nodes are available: 1 node(s) had untoler...,TBD,Review
1,FailedScheduling,trial0001,hpa,None,15:42,Warning,8,2026-10-09 15:11:02+00:00,2026-10-09 15:20:32+00:00,2026-10-09 14:55:19.548220+00:00,2026-10-09 15:25:25.356000+00:00,0 days 00:09:30,Pod,globeco-portfolio-service-8465f76dff-cjp8r,globeco,0/3 nodes are available: 1 node(s) had untoler...,TBD,Review
92,FailedGetResourceMetric,trial0002,keda,None,-3:45,Warning,7,2026-10-09 15:45:18+00:00,2026-10-09 16:00:19+00:00,2026-10-09 15:47:32.363742+00:00,2026-10-09 16:17:38.095000+00:00,0 days 00:15:01,HorizontalPodAutoscaler,globeco-portfolio-service-hpa,globeco,failed to get cpu utilization: unable to get m...,AUT-Induced,Retain
100,FailedScheduling,trial0002,keda,None,03:01,Warning,3,2026-10-09 15:50:34+00:00,2026-10-09 15:51:35+00:00,2026-10-09 15:47:32.363742+00:00,2026-10-09 16:17:38.095000+00:00,0 days 00:01:01,Pod,globeco-trade-service-7fb64d5fb4-pk46z,globeco,0/3 nodes are available: 1 node(s) had untoler...,TBD,Review
102,FailedScheduling,trial0002,keda,None,03:01,Warning,3,2026-10-09 15:50:34+00:00,2026-10-09 15:51:35+00:00,2026-10-09 15:47:32.363742+00:00,2026-10-09 16:17:38.095000+00:00,0 days 00:01:01,Pod,globeco-trade-service-7fb64d5fb4-vwpn5,globeco,0/3 nodes are available: 1 node(s) had untoler...,TBD,Review
96,FailedScheduling,trial0002,keda,None,03:31,Warning,2,2026-10-09 15:51:04+00:00,2026-10-09 15:51:34+00:00,2026-10-09 15:47:32.363742+00:00,2026-10-09 16:17:38.095000+00:00,0 days 00:00:30,Pod,globeco-pricing-service-64498f4df5-h9mgm,globeco,0/3 nodes are available: 1 node(s) had untoler...,TBD,Review


In [255]:
count = 0
for result in results:
    if result["reason"] == "FailedGetResourceMetric" and result["name"] != "globeco-portfolio-service-hpa":
        print(result["name"], result["trial_id"], result["count"], result["autoscaler"])
        count += 1
print(count)

0


In [256]:
autoscalers = ["keda", "hpa"]
for autoscaler in autoscalers:
    count = 0
    for result in results:
        if result["autoscaler"] == autoscaler:
            print(result["name"], result["trial_id"], result["count"], result["autoscaler"])
            count += 1
    print(f"{autoscaler}: {count}")


globeco-execution-service-6df778fb86-qrkk7 trial0002 1 keda
globeco-fix-engine-6888f4b86b-24cqk trial0002 1 keda
globeco-fix-engine-6888f4b86b-25wfx trial0002 1 keda
globeco-fix-engine-6888f4b86b-25wfx trial0002 1 keda
globeco-fix-engine-6888f4b86b-2j6mb trial0002 1 keda
globeco-fix-engine-6888f4b86b-2mdxs trial0002 1 keda
globeco-fix-engine-6888f4b86b-2mgdd trial0002 1 keda
globeco-fix-engine-6888f4b86b-2mgdd trial0002 1 keda
globeco-fix-engine-6888f4b86b-2sc97 trial0002 1 keda
globeco-fix-engine-6888f4b86b-47qw8 trial0002 1 keda
globeco-fix-engine-6888f4b86b-47qw8 trial0002 1 keda
globeco-fix-engine-6888f4b86b-486fm trial0002 1 keda
globeco-fix-engine-6888f4b86b-4hfmg trial0002 1 keda
globeco-fix-engine-6888f4b86b-4k5gf trial0002 1 keda
globeco-fix-engine-6888f4b86b-4nppl trial0002 1 keda
globeco-fix-engine-6888f4b86b-4tvkq trial0002 1 keda
globeco-fix-engine-6888f4b86b-4tvkq trial0002 1 keda
globeco-fix-engine-6888f4b86b-4w6rh trial0002 1 keda
globeco-fix-engine-6888f4b86b-54wwr tri

In [257]:
df = pd.DataFrame(results)
df = df.sort_values(by=["first_timestamp"])
df2 = df[df.name.str.startswith("globeco-portfolio-service")]
# display all the rows of the dataframe
with pd.option_context('display.max_rows', None):
    display(df2[df2.trial_id == "trial0002"])

,action,type,reason,count,first_timestamp,last_timestamp,trial_start_time,trial_end_time,duration,kind,name,namespace,message,trial_id,autoscaler
194,None,Warning,FailedGetResourceMetric,2,2026-10-08 15:01:22+00:00,2026-10-08 15:01:37+00:00,2026-10-08 14:42:09.530024+00:00,2026-10-08 15:12:15.562000+00:00,0 days 00:00:15,HorizontalPodAutoscaler,globeco-portfolio-service-hpa,globeco,failed to get memory utilization: did not rece...,trial0002,keda
195,None,Warning,FailedComputeMetricsReplicas,2,2026-10-08 15:01:22+00:00,2026-10-08 15:01:37+00:00,2026-10-08 14:42:09.530024+00:00,2026-10-08 15:12:15.562000+00:00,0 days 00:00:15,HorizontalPodAutoscaler,globeco-portfolio-service-hpa,globeco,"invalid metrics (2 invalid out of 2), first er...",trial0002,keda
193,None,Warning,FailedGetResourceMetric,3,2026-10-08 15:01:22+00:00,2026-10-08 15:01:52+00:00,2026-10-08 14:42:09.530024+00:00,2026-10-08 15:12:15.562000+00:00,0 days 00:00:30,HorizontalPodAutoscaler,globeco-portfolio-service-hpa,globeco,failed to get cpu utilization: did not receive...,trial0002,keda
191,None,Warning,Unhealthy,1,2026-10-08 15:01:25+00:00,2026-10-08 15:01:25+00:00,2026-10-08 14:42:09.530024+00:00,2026-10-08 15:12:15.562000+00:00,0 days 00:00:00,Pod,globeco-portfolio-service-8465f76dff-gvldr,globeco,"Startup probe failed: Get ""http://10.244.2.93:...",trial0002,keda
192,None,Warning,Unhealthy,1,2026-10-08 15:02:12+00:00,2026-10-08 15:02:12+00:00,2026-10-08 14:42:09.530024+00:00,2026-10-08 15:12:15.562000+00:00,0 days 00:00:00,Pod,globeco-portfolio-service-8465f76dff-t5jb6,globeco,"Startup probe failed: Get ""http://10.244.1.84:...",trial0002,keda


In [234]:
list(df[df.trial_id == "trial0002"].message)

['Startup probe failed: Get "http://10.244.2.24:8081/actuator/health/readiness": dial tcp 10.244.2.24:8081: connect: connection refused',
 'failed to get cpu utilization: did not receive metrics for targeted pods (pods might be unready)',
 'failed to get memory utilization: did not receive metrics for targeted pods (pods might be unready)',
 'invalid metrics (2 invalid out of 2), first error is: failed to get cpu resource metric value: failed to get cpu utilization: did not receive metrics for targeted pods (pods might be unready)',
 'Startup probe failed: Get "http://10.244.2.25:8082/actuator/health/readiness": dial tcp 10.244.2.25:8082: connect: connection refused']

In [195]:
len(df[df.reason == "FailedPreStopHook"])

0

In [196]:
df.groupby(["autoscaler", "reason"]).agg({"type": ["count"]})

type
                                        count
autoscaler reason                            
hpa        FailedComputeMetricsReplicas     1
           FailedGetResourceMetric          2
           Unhealthy                        2
keda       FailedComputeMetricsReplicas     1
           FailedGetResourceMetric          6
           Unhealthy                      168

In [197]:
df.groupby(["trial_id", "autoscaler", "reason"]).agg({"type": ["count"]})

type
                                                  count
trial_id  autoscaler reason                            
trial0001 keda       FailedComputeMetricsReplicas     1
                     FailedGetResourceMetric          6
                     Unhealthy                      168
trial0002 hpa        FailedComputeMetricsReplicas     1
                     FailedGetResourceMetric          2
                     Unhealthy                        2

In [ ]:
df = pd.DataFrame(results)
# order df by first_timestamp

df = df.sort_values(by=["first_timestamp"])
# display all the rows of the dataframe
with pd.option_context('display.max_rows', None):
    display(df[df.name.str.startswith("globeco-fix-engine-6888f4b86b-2rlpm")])

,action,type,reason,count,first_timestamp,last_timestamp,trial_start_time,trial_end_time,duration,kind,name,namespace,message,trial_id,autoscaler
2,None,Warning,Unhealthy,1,2026-10-07 20:27:19+00:00,2026-10-07 20:27:19+00:00,2026-10-07 20:14:16.577889+00:00,2026-10-07 20:44:22.294000+00:00,0 days,Pod,globeco-fix-engine-6888f4b86b-2rlpm,globeco,Startup probe failed: HTTP probe failed with s...,trial0001,keda
3,None,Warning,Unhealthy,1,2026-10-07 20:28:19+00:00,2026-10-07 20:28:19+00:00,2026-10-07 20:14:16.577889+00:00,2026-10-07 20:44:22.294000+00:00,0 days,Pod,globeco-fix-engine-6888f4b86b-2rlpm,globeco,"Readiness probe failed: Get ""http://10.244.3.7...",trial0001,keda


In [228]:
df2 = df[df.name.str.startswith("globeco-fix-engine-6888f4b86b-2rlpm")]
df2[["reason","count", "first_timestamp", "name", "message"]]

,reason,count,first_timestamp,name,message
2,Unhealthy,1,2026-10-07 20:27:19+00:00,globeco-fix-engine-6888f4b86b-2rlpm,Startup probe failed: HTTP probe failed with s...
3,Unhealthy,1,2026-10-07 20:28:19+00:00,globeco-fix-engine-6888f4b86b-2rlpm,"Readiness probe failed: Get ""http://10.244.3.7..."


In [193]:
fix_events = len(df[df.name.str.startswith("globeco-fix-engine")])
total_events = len(df)

print(f"Fix events: {fix_events}.  Other events {total_events-fix_events}")

Fix events: 163.  Other events 17


In [224]:
df[df.name.str.startswith("globeco-fix-engine-6888f4b86b-")].groupby("name").agg({"type": "count"})

,type
name,
globeco-fix-engine-6888f4b86b-27npb,1
globeco-fix-engine-6888f4b86b-2rlpm,2
globeco-fix-engine-6888f4b86b-4dcgk,1
globeco-fix-engine-6888f4b86b-4gckw,2
globeco-fix-engine-6888f4b86b-4j264,2
...,...
globeco-fix-engine-6888f4b86b-zdktg,1
globeco-fix-engine-6888f4b86b-zpfs2,2
globeco-fix-engine-6888f4b86b-zss4d,1
